In [10]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
MODEL_B = "distilbert-base-uncased"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)
MAX_LENGTH = 384
DOC_STRIDE = 96

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def prepare_qa_features(examples):
 tokenized = tokenizer_b(
 [q.strip() for q in examples["question"]],
 examples["context"],
 truncation="only_second",
 max_length=MAX_LENGTH,
 stride=DOC_STRIDE,
 return_overflowing_tokens=True,
 return_offsets_mapping=True,
 padding="max_length",
 )
 sample_mapping = tokenized.pop("overflow_to_sample_mapping")
 offsets = tokenized.pop("offset_mapping")
 start_positions, end_positions = [], []
 for feature_index, feature_offsets in enumerate(offsets):
  input_ids = tokenized["input_ids"][feature_index]
  cls_index = input_ids.index(tokenizer_b.cls_token_id)
  sequence_ids = tokenized.sequence_ids(feature_index)
  sample_index = sample_mapping[feature_index]
  answer_start = examples["answer_start"][sample_index]
  answer_end = answer_start + len(examples["answer_text"][sample_index])
  context_start = 0
  while sequence_ids[context_start] != 1:
    context_start += 1
    context_end = len(sequence_ids) - 1
  while sequence_ids[context_end] != 1:
    context_end -= 1
  if (
  feature_offsets[context_start][0] > answer_start
  or feature_offsets[context_end][1] < answer_end
  ):
    start_positions.append(cls_index)
    end_positions.append(cls_index)
    continue
  token_start = context_start
  while feature_offsets[token_start][1] <= answer_start:
    token_start += 1
  token_end = context_end
  while feature_offsets[token_end][0] >= answer_end:
    token_end -= 1
  start_positions.append(token_start)
  end_positions.append(token_end)
 tokenized["start_positions"] = start_positions
 tokenized["end_positions"] = end_positions
 return tokenized


In [12]:
from datasets import Dataset

In [13]:
dataset = Dataset.from_json(
    "/content/technical_support_qa_100_official_docs.json"
)

In [14]:
print(dataset)
print(dataset.column_names)
print(dataset[0])

Dataset({
    features: ['id', 'intent', 'question', 'context', 'answer_text', 'answer_start', 'source_title', 'source_url', 'source_section', 'source_type', 'context_style', 'retrieved_at', 'source_group', 'fact_group'],
    num_rows: 100
})
['id', 'intent', 'question', 'context', 'answer_text', 'answer_start', 'source_title', 'source_url', 'source_section', 'source_type', 'context_style', 'retrieved_at', 'source_group', 'fact_group']
{'id': 'QA001', 'intent': 'authentication', 'question': 'What is sufficient to use a bearer token?', 'context': 'A bearer token grants access based on possession, so it must be protected from disclosure.', 'answer_text': 'possession', 'answer_start': 38, 'source_title': 'RFC 6750 — OAuth 2.0 Bearer Token Usage', 'source_url': 'https://datatracker.ietf.org/doc/html/rfc6750', 'source_section': 'Bearer token semantics', 'source_type': 'official_documentation', 'context_style': 'faithful_paraphrase_of_source', 'retrieved_at': datetime.datetime(2026, 9, 17, 0

In [15]:
example = dataset[0]
start = example["answer_start"]
answer = example["answer_text"]

print(example["context"])
print()
print("Expected answer:", answer)
print("Extracted answer:",
      example["context"][start:start + len(answer)])

A bearer token grants access based on possession, so it must be protected from disclosure.

Expected answer: possession
Extracted answer: possession


In [16]:
split = dataset.train_test_split(
    test_size=0.30,
    seed=42
)

qa_train = split["train"]
qa_temp = split["test"]


temp_split = qa_temp.train_test_split(
    test_size=0.50,
    seed=42
)

qa_val = temp_split["train"]
qa_test = temp_split["test"]

In [17]:
qa_train_features = qa_train.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_train.column_names,
)

qa_val_features = qa_val.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_val.column_names,
)

qa_test_features = qa_test.map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_test.column_names,
)

In [9]:
from transformers import TrainingArguments, Trainer
from collections import Counter
import re
import string

def qa_tokens(text):
 text = text.lower().translate(str.maketrans("", "", string.punctuation))
 return re.sub(r"\b(a|an|the)\b", " ", text).split()

def score_qa_model(trainer, examples, features):
 # Match the feature preprocessing above, including overflow windows.
 encoded = tokenizer_b(
  [item["question"].strip() for item in examples],
  [item["context"] for item in examples],
  truncation="only_second", max_length=MAX_LENGTH, stride=DOC_STRIDE,
  return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
 )
 start_logits, end_logits = trainer.predict(features).predictions
 if len(start_logits) != len(encoded["overflow_to_sample_mapping"]):
  raise ValueError("QA prediction windows do not match tokenizer windows")
 best_answers = [""] * len(examples)
 best_scores = [float("-inf")] * len(examples)
 for feature_index, sample_index in enumerate(encoded["overflow_to_sample_mapping"]):
  valid = [i for i, part in enumerate(encoded.sequence_ids(feature_index)) if part == 1]
  starts = sorted(valid, key=lambda i: start_logits[feature_index][i], reverse=True)[:20]
  ends = sorted(valid, key=lambda i: end_logits[feature_index][i], reverse=True)[:20]
  offsets = encoded["offset_mapping"][feature_index]
  for start in starts:
   for end in ends:
    if end < start or end - start + 1 > 50:
     continue
    begin_char, end_char = offsets[start][0], offsets[end][1]
    if end_char <= begin_char:
     continue
    score = float(start_logits[feature_index][start] + end_logits[feature_index][end])
    if score > best_scores[sample_index]:
     best_scores[sample_index] = score
     best_answers[sample_index] = examples[sample_index]["context"][begin_char:end_char]
 exact_matches, token_f1s, long_context_errors = [], [], []
 for example, prediction in zip(examples, best_answers):
  predicted, expected = qa_tokens(prediction), qa_tokens(example["answer_text"])
  exact_matches.append(float(predicted == expected))
  overlap = sum((Counter(predicted) & Counter(expected)).values())
  token_f1s.append(2 * overlap / (len(predicted) + len(expected)) if predicted or expected else 1.0)
  if len(tokenizer_b(example["context"], add_special_tokens=False)["input_ids"]) > MAX_LENGTH and predicted != expected:
   long_context_errors.append({"question": example["question"], "expected": example["answer_text"], "predicted": prediction})
 return {"exact_match": sum(exact_matches) / len(exact_matches),
         "token_f1": sum(token_f1s) / len(token_f1s),
         "long_context_errors": long_context_errors}

args_b = TrainingArguments(
 output_dir="models/qa_model",
 learning_rate=3e-5,
 per_device_train_batch_size=8,
 per_device_eval_batch_size=8,
 num_train_epochs=10,
 eval_strategy="epoch",
 save_strategy="epoch",
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 report_to="none",
)
trainer_b = Trainer(
 model=model_b,
 args=args_b,
 train_dataset=qa_train_features,
 eval_dataset=qa_val_features,
)
# Fresh base checkpoint with a newly initialized QA head.
baseline_b = trainer_b.evaluate(qa_test_features, metric_key_prefix="baseline")
baseline_b.update(score_qa_model(trainer_b, qa_test, qa_test_features))
print("BASELINE B (test):", baseline_b)
trainer_b.train()
fine_tuned_b = trainer_b.evaluate(qa_test_features, metric_key_prefix="fine_tuned")
fine_tuned_b.update(score_qa_model(trainer_b, qa_test, qa_test_features))
print("FINE-TUNED B (same test):", fine_tuned_b)
trainer_b.save_model("models/qa_model")
tokenizer_b.save_pretrained("models/qa_model")

Training Loss,Validation Loss,Epoch
No log,5.941798,0


BASELINE B (test): {'baseline_loss': 5.941798210144043, 'exact_match': 0.0, 'token_f1': 0.10555555555555556, 'long_context_errors': []}


Epoch,Training Loss,Validation Loss
1,No log,4.055975
2,No log,3.014564
3,No log,2.506968
4,No log,2.201131
5,No log,1.713137
6,No log,1.572808
7,No log,1.414804
8,No log,1.366638
9,No log,1.384695
10,No log,1.422924


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Tuned Loss
No log,No log,10,1.845895


FINE-TUNED B (same test): {'fine_tuned_loss': 1.8458951711654663, 'exact_match': 0.5333333333333333, 'token_f1': 0.6222222222222222, 'long_context_errors': []}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/qa_model/tokenizer_config.json', 'models/qa_model/tokenizer.json')